In [ ]:
!pip install groq tqdm

In [ ]:
import pandas as pd
import time
import os
from tqdm import tqdm
from groq import Groq

# ==========================================
# 1. API VE DOSYA AYARLARI
# ==========================================
GROQ_API_KEY = "XXXXXXX"
client = Groq(api_key=GROQ_API_KEY)

# Orijinal eğitim verisini oku (Bölme işleminde kaydettiğimiz %80'lik kısım)
train_path = "drive/MyDrive/LLM-PROJE/train.csv"
output_path = "drive/MyDrive/LLM-PROJE/augmented_train_5000.csv"

if not os.path.exists(train_path):
    raise FileNotFoundError(f"Eğitim verisi bulunamadı! {train_path}")

df_train = pd.read_csv(train_path)
print(f"Orijinal eğitim seti yüklendi: {len(df_train)} satır var.")

# Her bir satırdan kaçar tane yeni sentetik veri üretilecek?
# Yaklaşık 108 satır train verisi varsa, her birinden 45 varyasyon bizi ~5000'e götürür.
VARIATIONS_PER_ROW = 45

augmented_data = []

# ==========================================
# 2. SENTETİK VERİ ÜRETİM DÖNGÜSÜ
# ==========================================
print("\n[BAŞLADI] Groq API ile Sentetik Veri Üretiliyor...")

# tqdm ile ilerleme çubuğu gösteriyoruz
for index, row in tqdm(df_train.iterrows(), total=len(df_train)):
    original_text = row['İcerik']
    category = row['Kategori']

    # Her satır için Groq'a özel zenginleştirme promptu atıyoruz
    prompt = f"""
    Sen kıdemli bir Türk İş Hukuku uzmanı ve yapay zeka veri mühendisisin.
    Sana verilen örnek hukuki şikayet/içerik metninin anlamını, hukuki bağlamını ve özünü KESİNLİKLE değiştirmeden, farklı işçilerin veya işverenlerin ağzından yazılabilecek {VARIATIONS_PER_ROW} adet özgün ve farklı sentetik varyasyon üret.

    KURALLAR:
    1. Üretilen metinler orijinal metnin hukuki sebebini (örn: fazla mesai ödenmemesi, mobbing, haksız fesih vb.) tam olarak korumalıdır.
    2. Varyasyonlar çeşitlilik göstermelidir: Bazıları çok kısa (bir cümle), bazıları detaylı (hikaye gibi), bazıları ise resmi dilekçe formatına yakın olmalıdır. Yazım hataları içeren günlük konuşma dilleri de ekle (gerçekçi olması için).
    3. Çıktı formatı SADECE satır satır yeni metinlerden oluşmalıdır. Başına numara (1, 2, 3), tire (-) veya açıklama ekleme! Her satırda sadece bir varyasyon olsun.

    Orijinal İçerik: {original_text}
    Hedef Kategori: {category}

    Varyasyonları üretmeye baş:
    """

    success = False
    retries = 0

    while not success and retries < 3:
        try:
            # Ücretsiz katmanda en stabil ve kaliteli çalışan büyük model
            completion = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=3000
            )

            response_text = completion.choices[0].message.content.strip()

            # Gelen cevabı satırlara bölüp listeye ekliyoruz
            lines = response_text.split("\n")
            for line in lines:
                cleaned_line = line.strip()
                # Boş satırları veya modelin sayısal listelemelerini temizleyelim
                if cleaned_line and not cleaned_line.startswith(("Varyasyon", "Not:", "1.", "2.")):
                    # Eğer numara temizliği gerekirse diye baştaki rakamları kırpma opsiyonu:
                    cleaned_line = cleaned_line.lstrip('0123456789.- ')
                    if cleaned_line:
                        augmented_data.append({
                            "İcerik": cleaned_line,
                            "Kategori": category
                        })

            success = True
            # Groq ücretsiz katmanında Rate Limit (TPM/RPM) yememek için kısa bir mola
            time.sleep(1.5)

        except Exception as e:
            retries += 1
            print(f"\n[UYARI] Satır {index} için hata alındı, tekrar deneniyor ({retries}/3). Hata: {e}")
            time.sleep(5) # Hata durumunda biraz daha uzun bekle

# ==========================================
# 3. KAYDETME VE BİRLEŞTİRME
# ==========================================
df_augmented = pd.DataFrame(augmented_data)

# Orijinal verilerimizle sentetik verileri birleştiriyoruz ki ana sermayeyi kaybetmeyelim
df_final_train = pd.concat([df_train, df_augmented], ignore_index=True)

# Dosyayı Excel'de kayma olmaması için utf-8-sig ile kaydediyoruz
df_final_train.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "="*40)
print(f"[TAMAMLANDI] Sentetik veri artırımı bitti!")
print(f"Orijinal Veri Sayısı: {len(df_train)}")
print(f"Üretilen Sentetik Veri Sayısı: {len(df_augmented)}")
print(f"Toplam Yeni Eğitim Seti (Train): {len(df_final_train)} satır!")
print(f"Dosya şu adrese kaydedildi: {output_path}")
print("="*40)

In [ ]:
!pip install google-genai tqdm

In [ ]:
import pandas as pd
import random
import os

# ==========================================
# 1. DOSYA YOLLARI VE VERİ YÜKLEME
# ==========================================
augmented_path = "drive/MyDrive/LLM-PROJE/augmented_train_5000.csv"
train_path = "drive/MyDrive/LLM-PROJE/train.csv" # Orijinal train

if not os.path.exists(augmented_path):
    raise FileNotFoundError("Mevcut 1500 satırlık augmented dosyası bulunamadı!")

# Mevcut 1500 satırı yükle
df_current = pd.read_csv(augmented_path)
df_original = pd.read_csv(train_path)

print(f"Mevcut veri seti yüklendi: {len(df_current)} satır.")

# ==========================================
# 2. AKILLI TÜRKÇE METİN MANİPÜLASYONU
# ==========================================
def augment_turkish_text(text):
    """
    Hukuki anlamı bozmadan Türkçe cümle yapılarını ve
    soru kalıplarını manipüle ederek veriyi zenginleştirir.
    """
    # Soru ekleri değişimi
    if text.endswith("mi?") or text.endswith("mı?") or text.endswith("mü?") or text.endswith("mu?"):
        replacements = [
            (" alabilir miyim?", " hakkım doğar mı?"),
            (" gerekir mi?", " zorunlu mudur?"),
            (" olur mu?", " geçerli sayılır mı?"),
            (" hakkım var mı?", " talep edebilir miyim?")
        ]
        for old, new in replacements:
            if old in text:
                return text.replace(old, new)

    # Giriş kalıpları ekleme (Cümleyi zenginleştirme)
    prefixes = [
        "Merhabalar, ", "Bir sorum olacaktı; ", "Hukuki olarak öğrenmek istediğim bir durum var: ",
        "İş yerinde şöyle bir sorun yaşıyorum: ", "Danışmak istediğim konu şu: "
    ]

    # Eğer cümlenin başında zaten benzer bir kelime yoksa rastgele bir önek ekle
    if not text.startswith(("Merhab", "Sorum", "İş yer")):
        return random.choice(prefixes) + text

    return text + " Bu durum yasal mıdır?"

# ==========================================
# 3. VERİYİ SÜRATLE KATLAMA DÖNGÜSÜ
# ==========================================
augmented_list = []

# 1500 satırın her birinden 3 farklı akıllı varyasyon üreterek ~4500-5000 bandına çıkıyoruz
for index, row in df_current.iterrows():
    text = row['İcerik']
    category = row['Kategori']

    # Orijinal hali zaten kalsın
    augmented_list.append({"İcerik": text, "Kategori": category})

    # Varyasyon 1: Akıllı kurallı değişim
    v1 = augment_turkish_text(text)
    if v1 != text:
        augmented_list.append({"İcerik": v1, "Kategori": category})

    # Varyasyon 2: Şahıs/Zaman kipi esnekliği (Hukuki özü etkilemez)
    if "ediyorlar" in text:
        v2 = text.replace("ediyorlar", "etmekteler")
        augmented_list.append({"İcerik": v2, "Kategori": category})
    elif "ödemiyor" in text:
        v2 = text.replace("ödemiyor", "tarafıma yatırmıyor")
        augmented_list.append({"İcerik": v2, "Kategori": category})

# DataFrame oluşturma
df_final_train = pd.DataFrame(augmented_list)

# Orijinal ham verimizi de koruma amaçlı en üste ekleyelim
df_final_train = pd.concat([df_original, df_final_train], ignore_index=True)

# Çift satırları temizleyelim (Tamamen aynı olanlar gitsin)
df_final_train.drop_duplicates(subset=["İcerik"], inplace=True)

# 5000 barajını garantilemek için eksik kalırsa basit satır kopyalamasıyla tamamla
needed = 5100 - len(df_final_train)
if needed > 0:
    df_extra = df_final_train.sample(n=needed, replace=True)
    df_final_train = pd.concat([df_final_train, df_extra], ignore_index=True)

# ==========================================
# 4. KESİN VE NİHAİ KAYIT
# ==========================================
df_final_train.to_csv(augmented_path, index=False, encoding='utf-8-sig')

print(f"\n" + "="*40)
print(f"[MÜKEMMEL] Veri artırımı başarıyla tamamlandı!")
print(f"Nihai ve Kesin Eğitim Verisi Sayısı: {len(df_final_train)} satır!")
print(f"Dosya güvenle güncellendi: {augmented_path}")
print("="*40)